# A first machine-learning example

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/y-rosenthal/yr-pre-ml/blob/main/jb/chapters/01-first-ml-example.ipynb)

Before we learn the tools, let's see where they lead. In this chapter, we build three small machine-learning
models that predict how much a customer will spend in a year. You do not need to understand every line yet: the
goal is to see the whole workflow once, from a data file to a prediction. Then we notice something important about
the code.

**In this chapter, you will learn to:**

- describe the machine-learning workflow in words: data, predictors, target, training, testing, error;
- run a complete example with three models, and compare their errors;
- see that most of the code prepares data with pandas and NumPy, which is what the rest of this book teaches.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

DATA_URL = "https://raw.githubusercontent.com/y-rosenthal/yr-pre-ml/main/data/"

The first cell imports the tools: `pandas` for data, and five tools from **scikit-learn** (imported as
`sklearn`), the most popular Python package for machine learning. Run it once before the other cells.

## The business question

A store chain has a loyalty program. For each member, it knows the age, the region, how many months the person has
been a customer, how often they visit, and how much they spend per visit. It also knows how much each member spent
last year.

The marketing team asks: **for a new member, how much will they spend in a year?** A good guess helps to decide how
much to invest in each customer.

**Machine learning (ML)** answers such questions from examples. We give the computer many customers whose spending
we know, and it finds a rule that connects the facts about a customer to the spending. Then we use the rule for
customers whose spending we do not know.

## Step 1: read the data

In [ ]:
customers = pd.read_csv(DATA_URL + "customers.csv")
customers.head()

> **In Thonny:** a script shows nothing for `customers.head()` on the last line. Write
> `print(customers.head())`. The chapter *Colab, Thonny and other IDEs* explains why.

The data is a table, called a **DataFrame** in pandas. Each row is one customer. Each column is one fact:

- `age`, `region`, `months_as_customer`, `visits_per_month`, `avg_basket` (the average amount spent per visit, in
  dollars): the facts we know about a customer;
- `annual_spend`: what the customer spent in a year, in dollars.

`head()` shows the first five rows. How many rows are there in all?

In [ ]:
customers.shape

500 rows (customers) and 6 columns.

## Step 2: choose the target and the predictors

The column we want to predict is the **target**. Here it is `annual_spend`. The columns we use to predict it are
the **predictors** (also called *features*). By custom, the predictors are named `X` and the target `y`:

In [ ]:
X = customers.drop(columns="annual_spend")
y = customers["annual_spend"]
X.shape

`X` has the 5 predictor columns, and `y` has the one target column.

## A first try fails

Let's try to fit a model now. `LinearRegression` is one of the simplest models. `fit` asks it to learn from `X`
and `y`:

In [ ]:
model = LinearRegression()
model.fit(X, y)

It fails. Read the **last line** of the error message: the model cannot convert a string such as `'South'` to a
number (`float`). Models work only with numbers, and the `region` column holds text.

That is not the only problem. Some values are missing. The first five rows above happen to be complete, but
other rows are not. pandas shows a missing value as `NaN` ("Not a Number"). Let's count the missing values in each
column:

In [ ]:
customers.isna().sum()

14 ages and 11 basket amounts are missing. `LinearRegression`, like many other models, also refuses missing
values.

**This is normal.** Real business data almost always has text columns and missing values. Before any model can
learn, someone has to prepare the data. That someone is you, with pandas.

## Step 3: prepare the data

We make two changes (the chapter *Preparing data for machine learning* explains them in detail):

1. Drop the rows that have a missing value. We keep most of the customers.
2. Replace the text column `region` with number columns: one column for each region, which is `True` (1) when the
   customer is in that region and `False` (0) when not.

In [ ]:
clean = customers.dropna()
clean = pd.get_dummies(clean, columns=["region"])
clean.shape

475 customers remain, and `region` became four columns (one for each region). Let's look at the new columns,
next to the target (a list of column names inside `[...]` selects those columns):

In [ ]:
clean[["annual_spend", "region_East", "region_North", "region_South", "region_West"]].head()

Now we choose `X` and `y` again, from the clean data:

In [ ]:
X = clean.drop(columns="annual_spend")
y = clean["annual_spend"]

## Step 4: split the data into training and test sets

How do we know whether a model is good? We cannot judge it on the customers it learned from: a model can simply
remember them. So we hide some customers from the model while it learns, and test it on them later.

`train_test_split` does this. It puts 75% of the rows in a **training set** and 25% in a **test set**, chosen at
random. `random_state=42` fixes the random choice, so that you get the same split as this book.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
X_train.shape, X_test.shape

356 customers to learn from, and 119 to test on.

## Step 5: train three models

We train three kinds of model on the same training set. Each one learns a different kind of rule:

- **linear regression** finds the best straight-line formula, such as "spend = a + b × visits + c × basket + ...";
- a **decision tree** learns a set of yes/no questions, such as "Is visits_per_month above 4.5?";
- a **random forest** builds many different decision trees and averages their answers.

`fit` is the step where a model learns from data. It is the same one line for every model. (Colab shows a short
description of the last model under the cell; you can ignore it.)

In [ ]:
linear = LinearRegression()
linear.fit(X_train, y_train)

tree = DecisionTreeRegressor(max_depth=5, random_state=42)
tree.fit(X_train, y_train)

forest = RandomForestRegressor(n_estimators=200, random_state=42)
forest.fit(X_train, y_train)

## Step 6: predict, and measure the error

Now each model predicts the spending of the 119 test customers, whom it has never seen. `predict` returns one
number for each customer:

In [ ]:
forest_predictions = forest.predict(X_test)
forest_predictions[:5]

Compare them with what these five customers really spent:

In [ ]:
y_test.head()

The predictions are close, but not exact. To summarize how close they are, we use the **mean absolute error
(MAE)**: for each customer, take the difference between the prediction and the real value, ignore the sign, and
average these differences over all test customers. The MAE is in dollars: "on average, the prediction is off by
this many dollars". Smaller is better.

In [ ]:
for name, model in [("linear regression", linear), ("decision tree", tree), ("random forest", forest)]:
    predictions = model.predict(X_test)
    error = mean_absolute_error(y_test, predictions)
    print(f"{name:18} MAE: ${error:,.0f}")

The three models make errors of different sizes. Here, the random forest predicts best. The single decision tree
predicts worst: with only five levels of yes/no questions, it can give only a limited number of different answers.
Linear regression is in between. A customer's yearly spending is roughly *visits per month × basket × 12*: two
numbers multiplied together. A straight-line formula adds numbers, so it can follow a multiplication only roughly.
The random forest averages many trees, and that gives smoother, closer predictions.

> **Try it:** change `max_depth=5` to `max_depth=2`, then run the training cell and the cell above again. Does the
> decision tree get better or worse?

## The important point

Look back at the code of this chapter and sort the lines into two groups:

| Lines that prepare the data (pandas, NumPy) | Lines that do machine learning (scikit-learn) |
|---|---|
| `pd.read_csv(...)`, `head()`, `shape` | `train_test_split(...)` |
| `isna().sum()`, `dropna()` | `LinearRegression()`, `DecisionTreeRegressor(...)`, `RandomForestRegressor(...)` |
| `pd.get_dummies(...)` | `fit(...)` |
| `drop(columns=...)`, `clean["annual_spend"]` | `predict(...)` |
| choosing `X` and `y`, slicing `[:5]` | `mean_absolute_error(...)` |

The machine-learning part is short, and it is almost the same for every model: create, `fit`, `predict`, measure.
**Most of the work, and most of the thinking, is in getting the data ready.** Data professionals often say that
they spend most of their time on this part.

The next chapters teach exactly that part: **NumPy**, the package for fast arrays of numbers (scikit-learn works on
NumPy arrays), and **pandas**, the package for tables of data. At the end of the book, you do the whole workflow
yourself, on new data.

## Summary

- **Machine learning** finds a rule that predicts a **target** (here `annual_spend`) from **predictors** (the other
  columns), using examples whose target is known.
- The workflow: read the data → prepare it (no text, no missing values) → choose `X` and `y` → split into training
  and test sets → `fit` models on the training set → `predict` the test set → compare the errors.
- **The MAE** (mean absolute error) is the average size of the prediction errors, in the units of the target.
- Models need numbers, and many models, such as `LinearRegression`, also refuse missing values. Preparing the
  data is most of the work, and it is done with pandas and NumPy.

## Further reading

- The scikit-learn "Getting Started" page: <https://scikit-learn.org/stable/getting_started.html>.

## Practice questions

### Question 1 (Easy)

In this chapter, what was the target, and what were the predictors?

<details class="yr-answer">
<summary>Answer</summary>

The target was `annual_spend`, the amount a customer spent in a year. The predictors were the other columns: `age`,
`region`, `months_as_customer`, `visits_per_month` and `avg_basket`. (After preparing the data, `region` became four
columns, one for each region.)

</details>

In [ ]:
# Write your answer here

### Question 2 (Easy)

Why did the first `model.fit(X, y)` fail? What second problem in the data would also have stopped it?

<details class="yr-answer">
<summary>Answer</summary>

The text column `region` stopped it: the models work only with numbers. The missing values (`NaN`) would also
stop `LinearRegression`. `dropna()` removed the rows with missing values, and `pd.get_dummies` replaced the text
column with number columns.

</details>

In [ ]:
# Write your answer here

### Question 3 (Easy)

Why do we test a model on customers that it did not see while it learned?

<details class="yr-answer">
<summary>Answer</summary>

A model can remember the customers it learned from, so it may look very good on them and still predict badly for
new customers. The test set shows how well the model predicts for customers it has never seen, which is the job we
want it to do.

</details>

In [ ]:
# Write your answer here

### Question 4 (Medium)

A model's MAE on the test set is 250. Explain in one sentence what that means for the marketing team.

<details class="yr-answer">
<summary>Answer</summary>

On average, the model's prediction of a customer's yearly spending is $250 too high or too low.

</details>

In [ ]:
# Write your answer here

### Question 5 (Medium)

Show the first five predictions of the **linear regression** model for the test customers, and the first five
real values next to them.

<details class="yr-answer">
<summary>Answer</summary>

Use the model named `linear`. `[:5]` takes the first five predictions, and `head()` the first five real values.

```python
print(linear.predict(X_test)[:5])
print(y_test.head())
```

</details>

In [ ]:
# Write your answer here

### Question 6 (Hard)

Split the data with `test_size=0.5` instead of `0.25` (keep `random_state=42`). Train a new random forest on the new
training set, and compute its MAE on the new test set. Is it bigger or smaller than before? Why might that be?

<details class="yr-answer">
<summary>Answer</summary>

Here the MAE is about $310, against about $250 before: bigger. With half of the data for testing, the forest
learns from only 237 customers instead of 356. (The test set changes too, so the numbers are not exactly
comparable.)

```python
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.5, random_state=42)
forest2 = RandomForestRegressor(n_estimators=200, random_state=42)
forest2.fit(X_train2, y_train2)
mean_absolute_error(y_test2, forest2.predict(X_test2))
```

</details>

In [ ]:
# Write your answer here

### Question 7 (Challenge)

Train a random forest that uses only two predictors, `visits_per_month` and `avg_basket`. Compare its MAE with the
forest that uses all the predictors. What does the result suggest about the other columns?

<details class="yr-answer">
<summary>Answer</summary>

A list of column names inside `[...]` selects those columns (the chapter *Series and DataFrames* explains it). The
forest with two predictors is clearly worse (an MAE of about $324, against about $250 with all the predictors).
So the other columns also carry information about spending: here, how long someone has been a customer, and the
region.

```python
two = ["visits_per_month", "avg_basket"]
small_forest = RandomForestRegressor(n_estimators=200, random_state=42)
small_forest.fit(X_train[two], y_train)
mean_absolute_error(y_test, small_forest.predict(X_test[two]))
```

</details>

In [ ]:
# Write your answer here